
# CPython Memory Management & Leak Diagnosis Demo

This notebook demonstrates:

- Reference counting
- Cyclic garbage collection
- Reference cycles
- Leak patterns
- `tracemalloc`
- `gc`
- `objgraph` usage
- Weak references
- Practical memory debugging workflow

Run the cells sequentially.



## 1. Reference Counting

CPython keeps a reference count for every object.
When the count becomes zero, the object is immediately destroyed.


In [ ]:

import sys

a = [1, 2, 3]

print("Initial refcount:", sys.getrefcount(a))

b = a
print("After assigning b = a:", sys.getrefcount(a))

c = a
print("After assigning c = a:", sys.getrefcount(a))

del b
print("After deleting b:", sys.getrefcount(a))

del c
print("After deleting c:", sys.getrefcount(a))



> `sys.getrefcount()` itself temporarily adds a reference, so the value appears one higher.



## 2. Deterministic Destruction

Objects are destroyed immediately when refcount reaches zero.


In [ ]:

class Resource:
    def __del__(self):
        print("Resource destroyed immediately")

r = Resource()

del r



## 3. Reference Cycles

Reference counting alone cannot reclaim cyclic references.


In [ ]:

import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.other = None

    def __del__(self):
        print(f"Deleting {self.name}")

a = Node("A")
b = Node("B")

a.other = b
b.other = a

del a
del b

print("Objects still exist because of cycle")

unreachable = gc.collect()

print("GC collected:", unreachable)



## 4. Generational Garbage Collection

CPython uses a generational collector.

- Generation 0: newest objects
- Generation 1: survivors from gen0
- Generation 2: long-lived objects


In [ ]:

import gc

print("GC thresholds:", gc.get_threshold())

print("GC counts:", gc.get_count())

print("GC stats:")
for i, stat in enumerate(gc.get_stats()):
    print(f"Generation {i}:")
    print(stat)



## 5. Manual GC Control


In [ ]:

import gc

gc.disable()
print("GC enabled?", gc.isenabled())

gc.enable()
print("GC enabled?", gc.isenabled())

collected = gc.collect()
print("Objects collected:", collected)



## 6. Simulating a Memory Leak

Global containers and caches can accidentally keep objects alive.


In [ ]:

leaky_cache = []

class BigObject:
    def __init__(self):
        self.data = [0] * 100000

for _ in range(50):
    leaky_cache.append(BigObject())

print("Objects retained in cache:", len(leaky_cache))



## 7. Using tracemalloc

`tracemalloc` helps identify where allocations come from.


In [ ]:

import tracemalloc

tracemalloc.start()

snapshot1 = tracemalloc.take_snapshot()

data = []
for _ in range(1000):
    data.append([0] * 1000)

snapshot2 = tracemalloc.take_snapshot()

stats = snapshot2.compare_to(snapshot1, 'lineno')

print("Top allocation differences:")

for stat in stats[:10]:
    print(stat)



## 8. Inspecting Objects with gc


In [ ]:

import gc

all_objects = gc.get_objects()

print("Tracked object count:", len(all_objects))

sample = all_objects[:5]

for obj in sample:
    print(type(obj))


In [ ]:

import gc

x = []
y = [x]

referrers = gc.get_referrers(x)

print("Number of referrers:", len(referrers))

for r in referrers[:3]:
    print(type(r))



## 9. Weak References

Weak references allow caches without preventing garbage collection.


In [ ]:

import weakref
import gc

class User:
    pass

cache = weakref.WeakValueDictionary()

u = User()

cache["alice"] = u

print("Before deletion:", dict(cache))

del u

gc.collect()

print("After deletion:", dict(cache))



## 10. objgraph (Optional Third-Party Tool)

Install:

```bash
pip install objgraph graphviz
```

Useful functions:

- `objgraph.show_most_common_types()`
- `objgraph.show_backrefs(obj)`
- `objgraph.find_backref_chain()`


In [ ]:

# Uncomment after installing objgraph

# import objgraph

# objgraph.show_most_common_types(limit=10)



## 11. Leak Diagnosis Workflow

Typical workflow:

1. Reproduce memory growth
2. Use `tracemalloc` snapshots
3. Inspect live objects with `gc`
4. Use `objgraph` for reference chains
5. Inspect caches/globals/closures
6. Investigate native extensions if Python allocations look normal



## 12. Native Memory Leaks

If memory grows but `tracemalloc` does not show increasing Python allocations:

- Suspect C extensions
- Use:
  - Valgrind
  - AddressSanitizer
  - massif
  - perf
  - top / ps / smem



# Summary

CPython memory management combines:

- Reference counting
- Generational garbage collection
- Cycle detection

Useful tools:

- `tracemalloc`
- `gc`
- `objgraph`
- `heapy`
- System profilers

Best practices:

- Avoid unintended global references
- Use weak references for caches
- Break cycles when needed
- Monitor allocations early
